In [3]:
2+2

4

In [6]:
%pip install -q openai pydantic tqdm requests

import json
from pathlib import Path

from openai import OpenAI
from pydantic import BaseModel
from tqdm import tqdm

Error connecting to agent: No such file or directory
You should consider upgrading via the '/speed-scratch/n_janchi/.jupyter-venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [7]:
ARTICLES_PATH = Path("LLM_classification_output_formatted.json")
ANSWERS_PATH = Path("w_q0_sparse_final_answers.json")
OUTPUT_PATH = Path("sentence_citation_labels3_v2.json")
MODEL_NAME = "ggml-org/gemma-4-26B-A4B-it-GGUF"
MAX_QUESTIONS = 3

In [8]:
client = OpenAI(
    api_key="--- IGNORE ---",
    base_url="http://localhost:11434/v1",
)

In [ ]:
class CitationLabel(BaseModel):
    justification: str
    label: int


def load_articles_by_uri(path: Path) -> dict[str, dict]:
    with path.open("r") as f:
        articles = json.load(f)
    return {str(article["uri"]): article["answer"] for article in articles}

In [ ]:

    def get_label(topic: str, question: str, sentence: str, abstract: str) -> CitationLabel:
        prompt = f"""
    Instruction: Classify the relationship between the SENTENCE and the PASSAGE
    using exactly ONE of THREE labels: supporting, contradicting, or irrelevant.

    Use these labels:
    - supporting (1): the passage contains information that helps support the sentence and is relevant to the patient's concern in the narrative.
    - contradicting (0): The passage provides information that conflicts with, disagrees with, refutes, or provides evidence against the sentence.
    - irrelevant (2): The passage does not meaningfully address the claim
    made in the sentence. It may be related to the general topic, but it
    does not provide evidence either supporting or contradicting the sentence.

    Important distinction:
    Do NOT classify a passage as contradicting merely because it is irrelevant,
    does not mention the sentence, or does not provide enough information to
    support the sentence. Contradiction requires evidence in the passage that
    conflicts with the claim made in the sentence.

    Rules:
    - Be strict and conservative.
    - supporting requires clear evidence or information consistent with the sentence.
    - contradicting requires clear evidence or information that conflicts with the sentence.
    - If the passage is unrelated to the claim, classify it as irrelevant.
    - Use only the provided information.
    - Return a short justification explaining the label choice.
    - Output exactly the requested structured response.

    Topic: {topic}
    Original Question: {question}

    Sentence:
    {sentence}

    PASSAGE:
    {abstract}
    """
        response = client.chat.completions.parse(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            response_format=CitationLabel,
        )
        return response.choices[0].message.parsed

In [ ]:
def build_sentence_citation_labels(max_questions: int = MAX_QUESTIONS) -> list[dict]:
    articles_by_uri = load_articles_by_uri(ARTICLES_PATH)

    with ANSWERS_PATH.open("r") as f:
        items = json.load(f)

    rows: list[dict] = []
    for item in tqdm(items[:max_questions], total=max_questions):
        for sentence_entry in item.get("response_sentences", []):
            sentence_text = sentence_entry.get("text", "").strip()
            references = item.get("references", [])

            for citation in references:
                citation_uri = str(citation)
                article = articles_by_uri.get(citation_uri)
                if article is None:
                    continue

                parsed = get_label(
                    topic=item.get("topic", ""),
                    question=item.get("question", ""),
                    sentence=sentence_text,
                    abstract=article,
                )

                rows.append(
                    {
                        "topic_id": item.get("topic_id"),
                        "topic": item.get("topic"),
                        "question": item.get("question"),
                        "sentence": sentence_text,
                        "citation": citation_uri,
                        "abstract": article,
                        "label": (
                            "supporting"
                            if parsed.label == 1
                            else "contradicting"
                            if parsed.label == 0
                            else "irrelevant"
                        ),
                        "justification": parsed.justification,
                    }
                )

    return rows

In [ ]:
rows = build_sentence_citation_labels()

100%|██████████| 3/3 [33:46<00:00, 675.52s/it]


In [ ]:
with OUTPUT_PATH.open("w") as f:
    json.dump(rows, f, indent=2)

print(f"Wrote {len(rows)} sentence-citation labels to {OUTPUT_PATH}")

Wrote 137 sentence-citation labels to sentence_citation_labels3_v2.json
